
**Tomilin Eugene 12533692**
# Exercise 3 (MapReduce in Practice) [3 points]
---

For this exercise, you are tasked with writing your own MapReduce program in Python and to
run it on the cluster on the provided datasets.

## Configuration

## How to Run This Notebook

**Execution Flow:**
Each task builds its MapReduce script incrementally using `%%file` magic commands. Cells must run sequentially to assemble the complete script.

**where it is placed :**

*Setup (run first):*
- Run Cell 4: Set `LOCAL_RUN` flag (True=local, False=cluster)
- Run Cell 5: for dependencies,  install mrjob + setuptools

*Task A:*
- Run Cells 8-21 sequentially (cleanup, build mymrjob1.py , execute)

*Task B:*
- Run Cells 23-38 sequentially (cleanup , build mymrjob2.py , execute)

**How It Works:**
- Cleanup cell: `rm -f mymrjob*.py` removes old script and data from old runs
- First build cell: `%%file mymrjob1.py` creates new script with imports/class
- Append cells: `%%file -a mymrjob1.py` add methods incrementally
- Runer cell: `!python3 mymrjob1.py {inputs}` executes created python , dumps output to file

In [146]:

LOCAL_RUN = True     # switches local and cluster mode
import os
os.environ['LOCAL_RUN'] = str(LOCAL_RUN)

HADOOP_STREAMING_JAR = '/usr/lib/hadoop/tools/lib/hadoop-streaming-3.3.6.jar'
os.environ['HADOOP_STREAMING_JAR'] = HADOOP_STREAMING_JAR

if LOCAL_RUN:
    posts = '../materials/data/posts.csv'
    users = '../materials/data/users.csv'
    comments = '../materials/data/comments.csv'
else:
    posts = '/home/adbs_shared/Ex_2/stackexchange/posts.csv'
    users = '/home/adbs_shared/Ex_2/stackexchange/users.csv'
    comments = '/home/adbs_shared/Ex_2/stackexchange/comments.csv'

In [147]:
import sys
import subprocess

try:
    import mrjob
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'mrjob'])

---

## Task a) Question-Answer-Comment Triples

Write a MapReduce job with "posts.csv" and "comment.csv" as inputs and following output:

    (user_questioned, user_answered, user_commented)

where each triple represents a complete question–answer–comment relationship: a user who asked a question, a user who answered that question, and a user who commented on the corresponding answer.

In "posts.csv", the user ids are stored in "owneruserid". Column "posttypeid" stores whether a post is a question (posttypeid = 1) or an answer (posttypeid=2). For answer posts, the column "parentid" stores the id of the corresponding question. In "comments.csv", the answer corresponding to the comment is stored in "postid" and the user who commented is stored in "userid" (You may ignore the comments where the "userid" is empty).

The output should begin with the following lines:

```
8,108,6
8,108,2750
8,319,2750
24,145,919
24,145,2669
24,930,1048
24,16188,919
117,61,117
117,61,61
28524,25392,28524
```

### Task a) - Implementation Notes

#### Pipeline

```mermaid
graph LR
    P["posts.csv"] --> M1
    C["comments.csv"] --> M1

    subgraph Step1["Step 1 - Join Questions with Answers"]
        M1["Mapper1: parse_all"]
        R1["Reducer1: join_qa"]
    end

    M1 -->|"key: question_id"| R1

    subgraph Step2["Step 2 - Join Q-A pairs with Comments"]
        M2["Mapper2: passthrough"]
        R2["Reducer2: join_comments"]
    end

    R1 -->|"key: answer_id"| M2
    M2 --> R2
    R2 --> O["output: user_questioned, user_answered, user_commented"]
```

#### Map/Reduce Sequence

**Step 1 — Parse and join questions with answers**

Mapper1 reads both CSV files: for questions emits `(question_id, ('Q', owner))` keyed by own id; for answers emits `(parent_id, ('A', answer_id, owner))` keyed by the question they answer; for comments emits `(post_id, ('C', user))` keyed by the post they comment on.

Reducer1 receives values grouped by key. When the key is a question_id it has Q and A records: it joins them and emits `(answer_id, ('QA', q_owner, a_owner))`. When the key is an answer_id (from comments) it passes the C records through unchanged. Comments on questions that have no matching answer are dropped later in Step 2.

**Step 2 — Join Q-A pairs with comments**

Mapper2 is an identity passthrough. Reducer2 receives values grouped by answer_id: QA pairs and C records arrive together, each comment produces one output triple `(q_owner, a_owner, c_owner)`.

#### I/O for map-reduce

```yaml
Mapper1:
  rule: tag and key rows from posts.csv and comments.csv for downstream joins
  posts.csv columns: [id, acceptedanswerid, answercount, ..., owneruserid, parentid, posttypeid, ...]
  comments.csv columns: [id, creationdate, postid, score, userdisplayname, userid]
  skip: header row with id column
  emit posts:
    question: key=post_id,  value=[Q, owner_id:int]
    answer:   key=parent_id, value=[A, post_id, owner_id:int]
  emit comments:
    comment:  key=post_id,   value=[C, user_id:int]

Reducer1:
  rule: join questions with their answers, pass comments through keyed by answer_id
  groups: values keyed by question_id contain [Q..., A..., C...] mixed
  join: for each answer under a question, emit [QA, q_owner, a_owner] keyed by answer_id
  passthrough: C records re-emitted under their original key unchanged

Mapper2:
  rule: identity passthrough, key is answer_id

Reducer2:
  rule: final join, comments matched to Q-A pairs produce triples
  groups: values keyed by answer_id contain [QA..., C...]
  output: for each C record paired with a QA record, emit csv line q_owner,a_owner,c_owner
  drop: comments without matching Q-A pair and Q-A pairs without comments
```

### Task a) - Cleanup

In [148]:
%%bash
rm -f mymrjob1.py
rm -f output_task_a*.txt
rm -rf output_task_a/

### Task a) - Step 1: Parse Posts and Comments (Mapper)

In [149]:
%%file mymrjob1.py

from mrjob.job import MRJob
from mrjob.step import MRStep
import mrjob.protocol       # importing task wrapper as in samples
import csv                  # csv stream parsing, line by line not like pandas
import logging

log = logging.getLogger(__name__)
LOG_UPD_SEC = 5

class MyMRJob1(MRJob):
    
    OUTPUT_PROTOCOL = mrjob.protocol.TextValueProtocol  # suppress JSON quoting
    
    def mapper1_parse_all(        # tag posts and comments, emit keyed for joins
            self, _, line         # line from csv, parsed row by row
            ):
        row = next(csv.reader([line]))  # csv.reader parses single line
        if row[0] == 'id':              # skip header, all three csvs start with id column
            return
        
        if len(row) >= 16:              # get row source by column count, posts.csv has 20 cols
            post_type = row[15]         # 1=question, 2=answer
            
            if post_type == '2' and row[14] and row[13]:  # answer with parent and owner
                yield row[14], ('A', row[0], int(float(row[13])))  # key=parent_id, .0 stripped
            elif post_type == '1' and row[13]:             # question with owner
                yield row[0], ('Q', int(float(row[13])))   # key=own post id
        
        elif len(row) >= 6:            # comments.csv has 6 cols
            if row[5]:                 # skip empty userid rows
                yield row[2], ('C', int(row[5]))  # key=post_id commented on

Writing mymrjob1.py


### Task a) - Step 2: Join Questions with Answers (Reducer)

In [150]:
%%file -a mymrjob1.py
    
    def reducer1_join_qa(          # join questions with answers, pass comments through
            self, question_id, values  # key is question_id or answer_id
            ):
        vals = list(values)        # combine to list for next steps
        questions = [v for v in vals if v[0] == 'Q']  # each is ('Q', owner_id)
        answers = [v for v in vals if v[0] == 'A']    # each is ('A', answer_id, owner_id)
        comments = [v for v in vals if v[0] == 'C']   # each is ('C', user_id)
        
        for comm in comments:      # comments pass through unchanged
            yield question_id, comm
        
        if not questions:          # break if no question found, nothing to join answers
            return
        
        q_owner = questions[0][1]  # [0]=first question tuple, [1]=owner field from ('Q', owner_id)
        self.increment_counter('Questions', 'Processed', 1)
        
        for ans in answers:
            answer_id = ans[1]                 # [1]=answer post id from ('A', answer_id, owner_id)
            answer_owner = ans[2]              # [2]=answer owner from ('A', answer_id, owner_id)
            yield answer_id, ('QA', q_owner, answer_owner)    # re-key by answer_id for step 2

Appending to mymrjob1.py


### Task a) - Step 3: Pass Through (Mapper)

In [151]:
%%file -a mymrjob1.py
    
    def mapper2_passthrough(       # empty mapper for MRStep, join is in reducer2
            self, key, value
            ):
        yield key, value

Appending to mymrjob1.py


### Task a) - Step 4: Join Comments with Q-A Pairs (Reducer)

In [152]:
%%file -a mymrjob1.py
    
    def reducer2_join_comments(    # match comments to Q-A pairs, emit triples
            self, answer_id, values):
        vals = list(values)      # combine to list
        qa_pairs = [v for v in vals if v[0] == 'QA']  # ('QA', q_owner, a_owner)
        comments = [v for v in vals if v[0] == 'C']   # ('C', user_id)
        
        if not qa_pairs or not comments:  # skip if either QA or Comment is not there
            return
        
        q_owner = qa_pairs[0][1]    # [1]=question owner from ('QA', q_owner, a_owner)
        a_owner = qa_pairs[0][2]    # [2]=answer owner from ('QA', q_owner, a_owner)
        
        for comm in comments:
            c_owner = comm[1]       # [1]=comment user from ('C', user_id)
            self.increment_counter('Triples', 'Generated', 1)    # logging
            yield None, f"{q_owner},{a_owner},{c_owner}"  # suppress "" 

Appending to mymrjob1.py


### Task a) - Step 5: Define MapReduce Pipeline

In [153]:
%%file -a mymrjob1.py
    
    def steps(self):
        return [
            MRStep(mapper=self.mapper1_parse_all,          # step 1: parse 
                   reducer=self.reducer1_join_qa),         # step 1: join Q-A
            MRStep(mapper=self.mapper2_passthrough,           # step 2: blank mapper
                   reducer=self.reducer2_join_comments)        # step 2: join with comments
        ]

if __name__ == '__main__':
    MyMRJob1.run()

Appending to mymrjob1.py


### Task a) - Run Job and Save Results

In [154]:
%%bash -s "$posts" "$comments"
TIMESTAMP=$(date +%Y%m%d_%H%M%S)
OUTPUT="output_task_a_${TIMESTAMP}.txt"
echo "=== Started at $(date +%T) ==="
if [ "$LOCAL_RUN" = "False" ]; then
    HDFS_OUT="/user/e12533692/task_a_output_${TIMESTAMP}"
    python3 mymrjob1.py -r hadoop --hadoop-streaming-jar "$HADOOP_STREAMING_JAR" \
        --output-dir "$HDFS_OUT" "$1" "$2"
    hdfs dfs -getmerge "$HDFS_OUT" - | sed 's/\t$//' > "$OUTPUT"  # Hadoop streaming appends \t between key and value
else
    python3 mymrjob1.py "$1" "$2" | tee "$OUTPUT"
fi
echo ""
echo "=== Results saved to $OUTPUT ==="

=== Started at 20:34:15 ===


No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob1.admin.20260510.183415.610998
Running step 1 of 2...

Counters: 1
	Questions
		Processed=2837

Running step 2 of 2...

Counters: 1
	Triples
		Generated=5437

job output is in /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob1.admin.20260510.183415.610998/output
Streaming final output from /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob1.admin.20260510.183415.610998/output...


364,1028,1909
2199,1764,887
2199,1764,930
2223,919,2223
2223,919,2223
2223,919,2223
2223,919,919
2224,1815,1815
2224,1815,1815
2224,1815,2224
2164,5792,919
2164,5792,919
2226,1709,2226
2226,1709,1709
2226,2168,2226
2226,2168,2456
2226,2168,2168
2226,2168,919
2226,2168,919
2226,2168,2168
2226,2168,919
18462,1709,18462
18462,1709,1709
18462,696,930
18462,930,1709
18462,930,696
18462,930,930
2221,930,2221
264,339,264
199,1909,199
199,1909,919
2228,364,2228
264,5,264
1124,1390,1124
1124,1390,1124
1124,1390,1390
1709,961,1709
529,2238,529
529,2238,529
2246,196,2246
2218,449,2218
2218,1390,2218
2218,1390,1050
2218,1390,1390
1005,887,1005
1808,1118,1679
1808,1118,449
1808,1118,1118
1808,1118,666
1808,1118,223
1808,1028,74
210,33,210
210,33,159
1808,74,223
1808,74,74
1808,74,223
1808,74,74
1808,74,223
1808,74,1670
1808,74,74
1808,74,1808
2091,1390,2091
2091,1390,930
2091,1390,485
2091,1390,1390
2091,1390,1390
1154,2084,1154
1808,2077,887
2261,930,2261
2261,930,930
1005,887,887
1005,887,1005
10

Removing temp directory /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob1.admin.20260510.183415.610998...



=== Results saved to output_task_a_20260510_203415.txt ===


---

## Task b) Top 100 Users by Reputation

Write a MapReduce job with "posts.csv" and "users.csv" as input and following output:

For each user compute the total number of question and answer posts they have created and output the 100 users with highest reputation.

Make sure the output has the following schema:

    displayname, i, reputation, question_count, answer_count

In "posts.csv", the user ids are stored in "owneruserid". Column "posttypeid" stores whether a post is a question (posttypeid = 1) or an answer (posttypeid=2). In "users.csv", the user ids are stored in "id" and their reputations are stored in "reputation".

In case that there are multiple users with the same reputation, you may resolve these conflicts randomly, i.e. pick one of them arbitrarily.

Use a combiner function where possible to reduce the communication cost.

Make sure that your program correctly deals with the header, data types, and possible sparse values.

In case of a correct implementation, the output should contain this text:

```
"Glen_b -Reinstate Monica",1,232889,18,4504
"whuber",2,231500,8,2377
"gung - Reinstate Monica",3,117552,12,1471
"Peter Flom - Reinstate Monica",4,87309,78,2814
"amoeba says Reinstate Monica",5,77182,43,344
```

### Task b) - Implementation Notes

#### Pipeline

```mermaid
graph LR
    P["posts.csv"] --> M1
    U["users.csv"] --> M1

    subgraph Step1["Step 1 - Join Posts with Users"]
        M1["Mapper1: parse_all"]
        C1["Combiner1: aggregate_posts"]
        R1["Reducer1: join_users_posts"]
    end

    M1 --> C1
    C1 -->|"key: user_id"| R1

    subgraph Step2["Step 2 - Global Top-K by Reputation"]
        M2["Mapper2: filter_topk"]
        MF["Mapper2_final"]
        R2["Reducer2: global_topk"]
    end

    R1 -->|"key: None"| M2
    M2 --> MF
    MF --> R2
    R2 --> O["output: displayname, rank, reputation, q_count, a_count"]
```

#### Map/Reduce Sequence

**Step 1 — Parse and join posts with users**

Mapper1 reads both CSV files: for posts emits `(owner_id, ('P', q_flag, a_flag))`; for users emits `(user_id, ('U', display_name, reputation))`. User IDs from posts.csv carry a .0 suffix in the dump, converted to integer at read time so keys match across both files.

Combiner1 locally aggregates post counts per user: sums q_flag and a_flag values and emits a single aggregated `('P', q_total, a_total)`. User records pass through unchanged. This reduces shuffle volume when a user has many posts.

Reducer1 receives combined P and U records per user_id. Joins them: extracts display_name and reputation from the U record, sums question and answer counts from P records. Emits `(None, (reputation, display_name, q_count, a_count))` — all keyed by None to force a single reduce group in Step 2.

**Step 2 — Global top-K by reputation**

Mapper2 maintains a local min-heap capped at K entries per instance, keeping the highest-reputation users seen so far. After processing all input splits, Mapper2_final yields the local top-K.

Reducer2 receives all mapper top-K outputs, uses `heapq.nlargest` to select the global top 100. Ranks them 1-based and formats output with CSV-safe name quoting for display names containing commas or double quotes.

#### I/O Specification

```yaml
Mapper1:
  rule: tag and type posts and users records, convert owneruserid from float to int
  posts.csv columns: [id, ..., owneruserid, parentid, posttypeid, ...]
  users.csv columns: [id, accountid, creationdate, displayname, downvotes, lastaccessdate, location, reputation, upvotes, views]
  skip: header row with id column
  emit posts:
    question: key=owner_id:int, value=[P, 1, 0]
    answer:   key=owner_id:int, value=[P, 0, 1]
  emit users:
    user:     key=user_id:int,  value=[U, display_name:str, reputation:int]

Combiner1:
  rule: local aggregation, sums post counts per user to cut shuffle volume
  groups: P records summed, U records forwarded as-is
  emit: key=user_id, value=[P, q_total, a_total] plus each [U, name, rep]

Reducer1:
  rule: join aggregated post counts with user profile
  groups: values keyed by user_id contain one U and possibly one aggregated P
  join: extract name and reputation from U, sum q_count and a_count from P
  output: key=None, value=[reputation, display_name, q_count, a_count]

Mapper2 / Mapper2_final:
  rule: local top-K filter using min-heap of size K=100, comparing by reputation
  heap: instance-level, lazily initialised to avoid cross-instance pollution
  emit after all records: top K entries from heap, keyed by None

Reducer2:
  rule: global top-K from mapper outputs, sort by reputation descending
  method: heapq.nlargest(K, values, key=lambda x: x[0])
  ties: resolved arbitrarily per heapq tie-breaking
  output: for each user in rank order, csv line name,rank,reputation,q_count,a_count
  quoting: names containing comma or double-quote are wrapped in double quotes
```

### Task b) - Cleanup

In [155]:
%%bash
rm -f mymrjob2.py
rm -f output_task_b*.txt
rm -rf output_task_b/

### Task b) - Step 1: Parse Posts and Users (Mapper)

In [156]:
%%file mymrjob2.py

from mrjob.job import MRJob
from mrjob.step import MRStep
import mrjob.protocol       # TextValueProtocol for plain output
import csv                  # line-by-line parsing
import heapq                # min-heap for top-K filtering
import logging
from datetime import datetime

log = logging.getLogger(__name__)
LOG_UPD_SEC = 5
K = 100

class MyMRJob2(MRJob):
    
    OUTPUT_PROTOCOL = mrjob.protocol.TextValueProtocol  # suppress quoting
    
    def mapper1_parse_all(        # tag posts and users, emit keyed by user_id for join
            self, _, line         # line from csv
            ):
        row = next(csv.reader([line]))  # csv.reader 
        if row[0] == 'id':              # skip header, both csvs start with id column
            return
        
        if len(row) >= 16:              # discriminate posts vs users by col count
            post_type = row[15]         # 1=question, 2=answer
            owner_id_raw = row[13]      # owneruserid
            
            if owner_id_raw:            # owner may be missing for community-wiki posts
                owner_id = int(float(owner_id_raw))  # .0 stripped, so it matches id type from users.csv 
                q_count = 1 if post_type == '1' else 0
                a_count = 1 if post_type == '2' else 0
                yield owner_id, ('P', q_count, a_count)  # key=int user id
        
        elif len(row) >= 8:            # users have 10 columns
            user_id = int(row[0])      # plain integer, no .0 suffix
            display_name = row[3]
            reputation = int(row[7] or '0')  # reputation missing for new users, fallback 0
            yield user_id, ('U', display_name, reputation)

Writing mymrjob2.py


### Task b) - Step 2: Aggregate Post Counts (Combiner)

In [157]:
%%file -a mymrjob2.py
    
    def combiner1_aggregate_posts(  # local sum of post counts, user records forwarded
            self, user_id, values
            ):
        vals = list(values)
        
        q_total = sum(v[1] for v in vals if v[0] == 'P')  # sum question flags
        a_total = sum(v[2] for v in vals if v[0] == 'P')  # sum answer flags
        
        if q_total > 0 or a_total > 0:     # emit aggregated P if user has any posts
            yield user_id, ('P', q_total, a_total)
        
        for v in vals:
            if v[0] == 'U':                # user profile records pass through unchanged
                yield user_id, v

Appending to mymrjob2.py


### Task b) - Step 3: Join Users with Posts (Reducer)

In [158]:
%%file -a mymrjob2.py
    
    def reducer1_join_users_posts(  # join user profile with post counts
            self, user_id, values
            ):
        vals = list(values)
        posts = [v for v in vals if v[0] == 'P']  # ('P', q_count, a_count)
        users = [v for v in vals if v[0] == 'U']  # ('U', display_name, reputation)
        
        if not users:              # user not found in users.csv, skip
            return
        
        display_name = users[0][1]  # [1]=name from ('U', name, rep)
        reputation = users[0][2]    # [2]=rep from ('U', name, rep)
        q_count = sum(p[1] for p in posts)  # [1]=q_count from ('P', qc, ac), sum across posts
        a_count = sum(p[2] for p in posts)  # [2]=a_count from ('P', qc, ac)
        self.increment_counter('Users', 'Processed', 1)
        
        yield None, (reputation, display_name, q_count, a_count)  # key=None forces single reduce group

Appending to mymrjob2.py


### Task b) - Step 4: Local TopK Filtering (Mapper)

In [159]:
%%file -a mymrjob2.py
    
    def mapper2_filter_topk(       # local top-K by reputation using min-heap
            self, _, value         # value is (rep, name, qc, ac) from reducer1
            ):
        # I took this trick from samples in tuwel, topK.py namely
        # The heap in mapper is used to optimize transfer volume.
        # Each mapper condenses tops in local heap , then pushes result, not row
        # Thus reducer gets at most K * num_mappers records, not the full user set

        if not hasattr(self, 'topk_heap'):  # lazy init, avoids class-variable sharing across instances
            self.topk_heap = []

        rep, name, qc, ac = value

        if len(self.topk_heap) < K:          # heap not yet full, push unconditionally
            heapq.heappush(self.topk_heap, (rep, name, qc, ac))
        elif rep > self.topk_heap[0][0]:     # new rep greater than heap min, replace
            heapq.heapreplace(self.topk_heap, (rep, name, qc, ac))

    def mapper2_final(self):       # emit local top-K after all input processed
        for item in self.topk_heap:
            yield None, item

Appending to mymrjob2.py


### Task b) - Step 5: Global TopK Selection (Reducer)

In [160]:
%%file -a mymrjob2.py
    
    def reducer2_global_topk(      # global top-K merge from all mapper outputs
            self, _, values
            ):
        vals = list(values)
        top_users = heapq.nlargest(K, vals, key=lambda x: x[0])  # global top-K by reputation
        self.increment_counter('TopUsers', 'Selected', len(top_users))
        log.info(f'Completed at {datetime.now().strftime("%H:%M:%S")} - Top {len(top_users)} users ranked')
        
        for i, (rep, name, qc, ac) in enumerate(top_users, 1):  # rank from 1
            if ',' in name or '"' in name:   # csv-safe quoting for names with special chars
                name = f'"{name}"'
            yield None, f"{name},{i},{rep},{qc},{ac}"

Appending to mymrjob2.py


### Task b) - Step 6: Define MapReduce Pipeline

In [161]:
%%file -a mymrjob2.py
    
    def steps(self):
        return [
            MRStep(mapper=self.mapper1_parse_all,              # step 1: join posts with users
                   combiner=self.combiner1_aggregate_posts,    # local aggregation of post counts
                   reducer=self.reducer1_join_users_posts),
            MRStep(mapper=self.mapper2_filter_topk,            # step 2: global top-K by reputation
                   mapper_final=self.mapper2_final,
                   reducer=self.reducer2_global_topk)
        ]

if __name__ == '__main__':
    MyMRJob2.run()

Appending to mymrjob2.py


### Task b) - Run Job and Save Results

In [162]:
%%bash -s "$posts" "$users"
TIMESTAMP=$(date +%Y%m%d_%H%M%S)
OUTPUT="output_task_b_${TIMESTAMP}.txt"
echo "=== Started at $(date +%T) ==="
if [ "$LOCAL_RUN" = "False" ]; then
    HDFS_OUT="/user/e12533692/task_b_output_${TIMESTAMP}"
    python3 mymrjob2.py -r hadoop --hadoop-streaming-jar "$HADOOP_STREAMING_JAR" \
        --output-dir "$HDFS_OUT" "$1" "$2"
    hdfs dfs -getmerge "$HDFS_OUT" - | sed 's/\t$//' > "$OUTPUT"  # Hadoop streaming appends \t between key and value
else
    python3 mymrjob2.py "$1" "$2" | tee "$OUTPUT"
fi
echo ""
echo "=== Results saved to $OUTPUT ==="

=== Started at 20:34:18 ===


No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob2.admin.20260510.183418.320509
Running step 1 of 2...

Counters: 1
	Users
		Processed=10000

Running step 2 of 2...
Completed at 20:34:20 - Top 100 users ranked

Counters: 1
	TopUsers
		Selected=100

job output is in /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob2.admin.20260510.183418.320509/output
Streaming final output from /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob2.admin.20260510.183418.320509/output...
Removing temp directory /var/folders/jm/9vywj7kj2t14mhd0gkkbtgh40000gn/T/mymrjob2.admin.20260510.183418.320509...


Glen_b -Reinstate Monica,1,232889,0,11
whuber,2,231500,3,335
gung - Reinstate Monica,3,117552,0,0
Peter Flom - Reinstate Monica,4,87309,2,44
Xi'an,5,68814,0,0
Frank Harrell,6,60786,0,18
Stephan Kolassa,7,60724,1,17
chl,8,46994,8,321
Rob Hyndman,9,44883,6,103
ttnphns,10,44117,2,3
AdamO,11,42871,0,0
kjetil b halvorsen,12,42026,0,0
Greg Snow,13,41868,0,19
Jeromy Anglim,14,39691,37,160
Dikran Marsupial,15,39049,1,81
Michael R. Chernick,16,37161,0,0
Has QUIT--Anony-Mousse,17,35494,0,0
Macro,18,35396,0,0
Dilip Sarwate,19,34220,0,0
Franck Dernoncourt,20,33534,0,0
mpiktas,21,31278,2,159
Reinstate Monica - G. Simpson,22,29979,1,46
cbeleites supports Monica,23,29048,0,1
jbowman,24,28342,0,0
Ben Bolker,25,27564,0,9
StasK,26,27300,0,0
IrishStat,27,26317,0,50
usεr11852,28,24233,0,0
Dimitriy V. Masterov,29,24057,0,0
cardinal,30,22903,0,40
Henry,31,22190,1,70
Robert Long,32,21511,0,0
Zach,33,21011,14,34
probabilityislogic,34,21007,6,148
user603,35,20195,12,69
Fomite,36,20174,0,0
John,37,19949,2,59
Ze

---
## **Your solution for Exercise 4 will consist of:**  
*  This notebook, filled with your solutions.